# 3. Building the knowledge base

**Learning objective:** see how raw documents become validated, retrievable
`SourceRecord` chunks, and why most of a RAG system's quality is decided here
rather than in the embedding model.

**Where this fits:** this is the offline half of the pipeline.

```
documents -> extract -> clean -> chunk -> SourceRecord list -> [embed -> index]
```

Nothing in this notebook is online, and nothing in it involves a language model.

## Retrieval quality starts before embeddings

If a chunk contains a navigation menu, a cookie banner, and half a sentence, no
embedding model can rescue it. The chunk will still be retrievable, it will still
be handed to the generator as evidence, and the generator will still try to use
it.

So this stage has three separate jobs, and they are kept separate on purpose:

1. **extraction** pulls text out of a file format, and nothing else
2. **cleaning** removes text that is definitely not content
3. **chunking** groups content into retrieval-sized units without breaking meaning

In [ ]:
import shutil
import tempfile
from pathlib import Path

from docx import Document

from mosaic_pathway.cleaning import (
    clean_paragraphs,
    is_noise_line,
    normalize_whitespace,
)
from mosaic_pathway.extraction import extract_paragraphs
from mosaic_pathway.knowledge_base import (
    MIN_RECORD_CHARS,
    OUTPUT_PATH,
    TARGET_CHUNK_CHARS,
    build_source_records,
    chunk_paragraphs,
)
from mosaic_pathway.models import SourceInventoryItem

print("knowledge base modules imported")

## A synthetic document to work on

The real Mosaic materials are private, so this notebook writes its own small
DOCX file into a temporary directory. It deliberately contains the same kinds of
problems the real exports have: headings, navigation chrome, a repeated
paragraph, a page footer, and blank lines between useful prose.

In [ ]:
BODY = [
    (
        "Many families begin by watching what already works in their week rather "
        "than designing a timetable in advance. A single repeated practice, kept "
        "for a month, tells you more about your family than a full plan that "
        "collapses in the second week of term."
    ),
    (
        "When a child returns to the same subject again and again, that return is "
        "information. Keeping a shared list of those returns gives you something "
        "concrete to build on when you are choosing the next book, outing, or "
        "conversation together."
    ),
    (
        "Rhythms are not schedules. A rhythm names the order of a day rather than "
        "the clock time of each part, which leaves room for a slow morning or a "
        "long walk without the whole day being counted as a failure."
    ),
    (
        "Time outdoors tends to be the first thing lost and the easiest thing to "
        "restore. A short walk before the day begins costs very little and often "
        "changes the tone of everything that follows it."
    ),
    (
        "Community for a self-directed family is usually built from small, low "
        "pressure contacts rather than one large commitment. One informal meetup "
        "a month is a reasonable starting point for most families."
    ),
    (
        "Progress in the first year is easier to see in relationships than in "
        "output. Noticing that a child asked a question they would not have asked "
        "before is a fair measure of how the year is going."
    ),
]

temp_dir = Path(tempfile.mkdtemp(prefix="mosaic-nb03-"))
docx_path = temp_dir / "synthetic-guide.docx"

document = Document()
document.add_heading("Rhythms for a gentle week", level=1)
document.add_paragraph("Home")
document.add_paragraph("Skip to content")
document.add_paragraph("")
document.add_paragraph(BODY[0])
document.add_paragraph(BODY[1])
document.add_paragraph(BODY[1])
document.add_heading("Outdoors and community", level=2)
document.add_paragraph(BODY[2])
document.add_paragraph("Page 2 of 4")
document.add_paragraph(BODY[3])
document.add_paragraph(BODY[4])
document.add_paragraph("Join our mailing list today")
document.add_paragraph(BODY[5])
document.add_paragraph("Subscribe")
document.add_paragraph("© 2026 Synthetic Publisher")
document.save(str(docx_path))

print("wrote", docx_path.name)

## Extraction: format handling only

`extract_paragraphs` dispatches on the file suffix and returns raw paragraphs in
document order. It makes no judgement about what is worth keeping, which is why
the empty lines and the menu items are still here.

In [ ]:
raw_paragraphs = extract_paragraphs(docx_path)

print("raw paragraphs:", len(raw_paragraphs))

for paragraph in raw_paragraphs[:6]:
    print(repr(paragraph[:60]))

## Cleaning: conservative on purpose

The cleaning rules are deliberately dull. There is an explicit list of exact
navigation lines, a handful of footer patterns, whitespace normalization, and a
consecutive-duplicate filter. That is all.

Aggressive cleaning is tempting and dangerous: a clever heuristic that drops
"short lines" would also drop a real one-sentence paragraph, and nobody would
notice until a family got a pathway missing the point of a document.

In [ ]:
print(normalize_whitespace("  Rhythms   are\n\tnot schedules.  "))
print()

for candidate in [
    "Home",
    "Page 2 of 4",
    "© 2026 Synthetic Publisher",
    "Rhythms are not schedules.",
]:
    print(f"{is_noise_line(candidate)!s:<6} {candidate}")

In [ ]:
cleaned_paragraphs = clean_paragraphs(raw_paragraphs)

print("raw paragraphs    :", len(raw_paragraphs))
print("cleaned paragraphs:", len(cleaned_paragraphs))
print()

removed = [
    paragraph
    for paragraph in (normalize_whitespace(item) for item in raw_paragraphs)
    if paragraph and paragraph not in cleaned_paragraphs
]

print("removed as noise:", removed)

Two things are worth noticing.

The headings survived. They are short, but they are content: they tell a reader
and a retriever what the surrounding paragraphs are about.

`Join our mailing list today` also survived. It is obviously noise to a human and
completely invisible to a rule set built from exact matches. That is the accepted
trade-off: unknown noise leaks through, and known content is never destroyed.

## Chunking: paragraph-aware, with a small carried overlap

Chunks are built by accumulating whole paragraphs until a target size is reached,
never by cutting at a fixed character count. A paragraph longer than the maximum
is split on word boundaries as a last resort.

When a chunk is flushed, its final paragraph is carried into the next chunk if it
is short enough. That overlap keeps a sentence that ends one chunk from losing
the sentence that answers it.

In [ ]:
default_chunks = chunk_paragraphs(cleaned_paragraphs)
small_chunks = chunk_paragraphs(cleaned_paragraphs, target_chars=600, max_chars=800)

print(
    f"target {TARGET_CHUNK_CHARS} chars -> {len(default_chunks)} chunks, sizes {[len(chunk) for chunk in default_chunks]}"
)
print(
    f"target  600 chars -> {len(small_chunks)} chunks, sizes {[len(chunk) for chunk in small_chunks]}"
)
print()
print("first 80 characters of chunk 2:", small_chunks[1][:80])

## From chunks to records: metadata and lineage

A chunk on its own is just text. `build_source_records` attaches the inventory
metadata that makes it usable evidence: a stable id, the document title, the
originating filename, the topics, and a mapped content and authority type.

The inventory is a human-reviewed file. Deciding that a podcast transcript is
lived experience rather than framework guidance is an editorial judgement, so it
is made once, by a person, and recorded rather than inferred at build time.

In [ ]:
inventory_item = SourceInventoryItem(
    source_id="synthetic-guide",
    filename="synthetic-guide.docx",
    title="Rhythms for a gentle week",
    format="docx",
    content_type="mixed",
    primary_topics=["rhythm", "outdoor learning", "community"],
    audience="parents new to self-directed learning",
    rag_priority="core",
    authority_type="expert_guidance",
    requires_cleaning=True,
    notes="Synthetic stand-in used for the notebook series.",
)

records = build_source_records(inventory_item, cleaned_paragraphs)

for record in records:
    print(
        record.source_id,
        "|",
        record.content_type,
        "|",
        record.authority_type,
        "|",
        len(record.text),
        "chars",
    )

print()
print("topics carried from the inventory:", records[0].topics)
print("originating file:", records[0].source_file)

Record ids are `{source_id}-{index:04d}`. They are stable for a given document
and a given chunking configuration, which matters because those ids end up in
the vector index, in generated citations, and in the retrieval evaluation set.

Change the chunk size and the ids shift underneath all three. That is a real
migration, not a tuning knob.

## Failure path: chunks that are too small to be evidence

Anything shorter than `MIN_RECORD_CHARS` is dropped rather than indexed. A
three-word fragment retrieves badly and reads worse when handed to a generator as
supporting evidence.

In [ ]:
fragment_records = build_source_records(inventory_item, ["Rhythms."])

print("minimum record length:", MIN_RECORD_CHARS)
print("records built from a single fragment:", len(fragment_records))

In [ ]:
shutil.rmtree(temp_dir, ignore_errors=True)

print("temporary directory removed:", not temp_dir.exists())

## Why the processed output stays private

`build_knowledge_base` writes every record to `data/processed/source_records.json`.
That file is a complete, machine-readable copy of the private Mosaic materials,
so the whole `data/processed/` directory is ignored by Git, and nothing in this
notebook prints its contents.

The cell below reports structure only, and it is disabled by default so that a
default run of this notebook never touches private data.

In [ ]:
INSPECT_LOCAL_KNOWLEDGE_BASE = False

if not INSPECT_LOCAL_KNOWLEDGE_BASE:
    print(
        "Skipped: set INSPECT_LOCAL_KNOWLEDGE_BASE to True to report local aggregates."
    )
elif not OUTPUT_PATH.is_file():
    print(
        "No local knowledge base found. Build one with: uv run python -m mosaic_pathway.knowledge_base"
    )
else:
    import json
    from collections import Counter

    payload = json.loads(OUTPUT_PATH.read_text(encoding="utf-8"))
    lengths = [len(item["text"]) for item in payload]
    counts = Counter(item["source_id"].rsplit("-", 1)[0] for item in payload)

    print("records:", len(payload))
    print("documents:", len(counts))
    print("records per document:", dict(counts))
    print(
        f"chunk length min/avg/max: {min(lengths)} / {sum(lengths) // len(lengths)} / {max(lengths)}"
    )

## Key takeaways

* Extraction, cleaning, and chunking are separate steps with separate failure
  modes, and keeping them separate is what makes each one testable.
* Cleaning is conservative by design: it removes known noise and never guesses.
  Unknown noise survives, and that is the safer error.
* Chunking follows paragraphs and carries a short overlap, so retrievable units
  stay readable.
* Metadata and lineage come from a human-reviewed inventory, not from inference.
* Record ids are stable and load-bearing; changing chunk sizes is a migration.
* The processed output is a copy of private material and is never committed.

## Next

Notebook 4 turns these records into vectors, stores them in a local Qdrant
collection, and measures whether the right ones come back.